# Course 4: Large Language Models
## Session 5: Post-training I: fine-tuning with LoRA and QLoRA

**Goal:** freeze a 4-bit base model, train small LoRA adapters on instructions, and test on instructions the model has not seen.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

In [ ]:
cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=64)
base = train_lm(cfg)
print("base model trained | validation loss %.3f | valid sentences in 200 samples: %.2f" % (val_loss(base), sum(map(is_valid, sample_sentences(base))) / 200))

### 1. Instruction data
The base model has only read plain sentences. Now teach it a question-and-answer format:

`প্রশ্ন : মা ভাত কী করে ? উত্তর : খায় ।`

The answer needs knowledge the base model already has (which verb goes with which object). Two subjects and two objects are held out of the instruction data, giving an easy test and a hard one.

In [ ]:
def instruction(subj, obj): return ["প্রশ্ন", ":", subj, obj, "কী", "করে", "?", "উত্তর", ":", VERB_OF[obj], "।"]
objs = [o for o, _ in OBJ_VERB]; held_out = objs[-2:]
train_q = [instruction(s, o) for s in SUBJ[:6] for o in objs[:-2]]
test_easy = [instruction(s, o) for s in SUBJ[6:] for o in objs[:-2]]        # new subjects, objects seen in training
test_q = [instruction(s, o) for s in SUBJ for o in held_out]                # objects never seen in any instruction
print(" ".join(train_q[0])); print("training instructions:", len(train_q), "| held-out subjects:", SUBJ[6:], "| held-out objects:", held_out)
Xq = torch.tensor([enc(q) for q in train_q])

@torch.no_grad()
def answer_accuracy(model, questions):
    x = torch.tensor([enc(q) for q in questions])
    pred = model(x[:, :9])[0][:, -1].argmax(-1)                    # the token after "উত্তর :"
    return (pred == x[:, 9]).float().mean().item()
print("base model, before any fine-tuning: accuracy on new subjects %.2f | on held-out objects %.2f" % (answer_accuracy(base, test_easy), answer_accuracy(base, test_q)))

### 2. QLoRA: a 4-bit frozen base plus trainable low-rank adapters
The base weights are quantized to 4 bits and never updated. Next to each weight matrix we add `B A` with rank 4, in full precision, and train only that.

In [ ]:
import copy
class LoRALinear(nn.Module):
    def __init__(self, base_layer, r=4, alpha=8):
        super().__init__()
        self.base = base_layer
        self.A = nn.Parameter(torch.randn(r, base_layer.in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, r))       # zero at the start: the model is unchanged
        self.scale = alpha / r
    def forward(self, x): return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

def add_lora(model, r=4):
    m = copy.deepcopy(model)
    for p in m.parameters(): p.requires_grad_(False)                         # freeze everything
    for blk in m.blocks: blk.qkv, blk.proj, blk.fc, blk.out = LoRALinear(blk.qkv, r), LoRALinear(blk.proj, r), LoRALinear(blk.fc, r), LoRALinear(blk.out, r)
    return m

In [ ]:
def quantize_(model, bits=4):
    levels = 2 ** (bits - 1) - 1
    with torch.no_grad():
        for blk in model.blocks:
            for layer in (blk.qkv, blk.proj, blk.fc, blk.out):
                w = layer.weight; scale = w.abs().amax(dim=1, keepdim=True) / levels
                w.copy_(torch.round(w / scale).clamp(-levels, levels) * scale)
    return model

def finetune(model, steps=300, lr=5e-3):
    params = [p for p in model.parameters() if p.requires_grad]; opt = torch.optim.AdamW(params, lr=lr)
    for _ in range(steps):
        b = Xq[torch.randint(len(Xq), (32,))]
        y = b[:, 1:].clone(); y[:, :8] = -100                      # train on the answer only, not on the question
        _, loss = model(b[:, :-1], y); opt.zero_grad(); loss.backward(); opt.step()
    return model

torch.manual_seed(0)
q_base = quantize_(copy.deepcopy(base), 4)
qlora = finetune(add_lora(q_base))
n_train = sum(p.numel() for p in qlora.parameters() if p.requires_grad); n_all = sum(p.numel() for p in qlora.parameters())
print(f"trainable: {n_train} of {n_all} parameters ({100 * n_train / n_all:.1f}%)")
r_qlora = (answer_accuracy(qlora, train_q), answer_accuracy(qlora, test_easy), answer_accuracy(qlora, test_q))
print("QLoRA: accuracy on training instructions %.2f | new subjects %.2f | held-out objects %.2f" % r_qlora)
assert r_qlora[0] > 0.95

### 3. Two comparisons
- Full fine-tuning: update every weight of the full-precision model.
- LoRA on a model that never saw the plain sentences: the format can be learned, but there is no knowledge to reuse.

In [ ]:
torch.manual_seed(0)
full = copy.deepcopy(base); full = finetune(full, lr=1e-3)
r_full = (answer_accuracy(full, train_q), answer_accuracy(full, test_easy), answer_accuracy(full, test_q))
blank = add_lora(GPT(cfg)); blank = finetune(blank)
r_blank = (answer_accuracy(blank, train_q), answer_accuracy(blank, test_easy), answer_accuracy(blank, test_q))
print("                                  training   new subjects   held-out objects")
for name, r in [("QLoRA on the pretrained base", r_qlora), ("full fine-tuning", r_full), ("LoRA on an untrained base", r_blank)]:
    print(f"{name:34s} {r[0]:8.2f} {r[1]:14.2f} {r[2]:18.2f}")
print("chance level on held-out objects: about %.2f" % (1 / 8))

Read the two test columns separately. New subjects with familiar objects are easy: the format carries over. Objects that never appeared in an instruction are a different matter: the adapter has only ever produced six answers, so it has learned a mapping among those six and does not reach back into what the base model knows about the other two. Here even the untrained base matches the pretrained one, which tells you this toy task is too easy to show the value of pretraining. Evaluation sets must include the hard kind of held-out case, or every method looks perfect.

### 4. What fine-tuning did to the old skill
Does the fine-tuned model still write plain sentences? Full fine-tuning can overwrite old behaviour (catastrophic forgetting). Adapters can simply be switched off.

In [ ]:
print("validation loss on plain sentences:  base %.3f | QLoRA %.3f | full fine-tuning %.3f" % (val_loss(base), val_loss(qlora), val_loss(full)))

### 5. QLoRA on a real model with Bangla instructions (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU and `pip install transformers peft bitsandbytes datasets accelerate`.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
name = "<a small open model, 1 to 3 billion parameters>"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, device_map="auto",
          quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16))
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))   # names differ between models: print(model) to check
model.print_trainable_parameters()
# Data: 2,000 to 10,000 Bangla instruction-response pairs. Format each with the model's own chat template (tok.apply_chat_template).
# Mask the loss on the instruction tokens, train for 1 to 3 epochs with learning rate 2e-4, and hold out 200 pairs for evaluation.

### TODO
1. Repeat section 2 with ranks 1, 2, 8 and with 5 seeds. Plot held-out accuracy with error bars.
2. Put LoRA on the attention matrices only. What changes?
3. Run the real QLoRA cell and compare 20 answers before and after fine-tuning by hand, using a written rubric.